In [0]:
from pyspark.sql import functions as f
from pyspark.sql import types as t
from datetime import date, timedelta, datetime
from zoneinfo import ZoneInfo
import requests

def parametro(nome, padrao):
    try:
        return dbutils.widgets.get(nome)
    except Exception:
        dbutils.widgets.text(nome, padrao)
        return dbutils.widgets.get(nome)

catalogo = parametro("catalogo", "workspace")
spark.sql(f"use catalog `{catalogo.replace('`', '``')}`")

spark.sql("create schema if not exists bronze")

base_path = parametro("base_path", "/Volumes/workspace/default/movies_data/").rstrip("/") + "/"
data_inicio = parametro("data_inicio", "").strip()
data_fim = parametro("data_fim", "").strip()

hoje = datetime.now(ZoneInfo("America/Sao_Paulo")).date()
inicio = datetime.strptime(data_inicio, "%m-%d-%Y").date() if data_inicio else hoje - timedelta(days=7)
fim = datetime.strptime(data_fim, "%m-%d-%Y").date() if data_fim else hoje

if inicio > fim:
    raise ValueError("data_inicio deve ser anterior ou igual a data_fim")

In [0]:
encontrados = {x.name for x in dbutils.fs.ls(base_path)}
candidatos = ["movies_info_TMDB_IMDB.csv", "movies_info_IMDB_TMDB.csv"]
arquivo_info = next((x for x in candidatos if x in encontrados), None)
if arquivo_info is None:
    raise FileNotFoundError(f"CSV de metadados ausente em {base_path}")

arquivos = {
    arquivo_info: "bronze.tb_movies_info",
    "movies_financials_IMDB_TMDB.csv": "bronze.tb_movies_financials",
    "movies_metrics_IMDB_TMDB.csv": "bronze.tb_movies_metrics",
    "credits_and_tags_IMDB_TMDB.csv": "bronze.tb_credits_and_tags",
    "movies_reviews.csv": "bronze.tb_movies_reviews"
}
faltando = set(arquivos) - encontrados
if faltando:
    raise FileNotFoundError(f"Arquivos ausentes: {sorted(faltando)}")

for arquivo, tabela in arquivos.items():
    df = (spark.read.format("csv")
          .option("header", "true").option("inferSchema", "false")
          .option("mode", "PERMISSIVE").option("multiLine", "true")
          .option("quote", '"').option("escape", '"')
          .load(base_path + arquivo)
          .withColumn("ingestion_datetime", f.current_timestamp()))
    df.write.format("delta").mode("append").saveAsTable(tabela)


In [0]:
endpoint = (
    "https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/"
    "CotacaoDolarPeriodo(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)"
)

def consultar_cotacoes(inicio_consulta, fim_consulta):
    params = {
        "@dataInicial": f"'{inicio_consulta.strftime('%m-%d-%Y')}'",
        "@dataFinalCotacao": f"'{fim_consulta.strftime('%m-%d-%Y')}'",
        "$select": "dataHoraCotacao,cotacaoCompra", "$format": "json",
       
    }
    registros = []
    url = endpoint
    while url:
        resposta = requests.get(url, params=params, timeout=60)
        resposta.raise_for_status()
        payload = resposta.json()
        registros.extend(payload.get("value", []))
        url = payload.get("@odata.nextLink") or payload.get("odata.nextLink")
        params = None
    return registros

dados = consultar_cotacoes(inicio - timedelta(days=30), fim)
if not dados:
    raise ValueError("API sem cotação no intervalo e na janela anterior de 30 dias")
schema = t.StructType([
    t.StructField("dataHoraCotacao", t.StringType(), True),
    t.StructField("cotacaoCompra", t.DoubleType(), True)
])
registros = [(x.get("dataHoraCotacao"), float(x["cotacaoCompra"]) if x.get("cotacaoCompra") is not None else None)
             for x in dados]
df_cotacao = spark.createDataFrame(registros, schema).withColumn("ingestion_datetime", f.current_timestamp())
df_cotacao.write.format("delta").mode("append").saveAsTable("bronze.tb_cotacao_dolar")
display(df_cotacao.orderBy("dataHoraCotacao"))


dataHoraCotacao,cotacaoCompra,ingestion_datetime
2026-09-08 13:02:38.447299,5.085,2026-10-16T01:58:18.333Z
2026-09-09 13:11:35.346979,5.0973,2026-10-16T01:58:18.333Z
2026-09-10 13:09:28.70012,5.1143,2026-10-16T01:58:18.333Z
2026-09-11 13:07:22.532196,5.0912,2026-10-16T01:58:18.333Z
2026-09-14 13:10:08.144425,5.169,2026-10-16T01:58:18.333Z
2026-09-15 13:09:19.199664,5.1484,2026-10-16T01:58:18.333Z
2026-09-16 13:05:30.35873,5.152,2026-10-16T01:58:18.333Z
2026-09-17 13:03:21.858212,5.1515,2026-10-16T01:58:18.333Z
2026-09-18 13:03:34.742036,5.1569,2026-10-16T01:58:18.333Z
2026-09-21 13:06:51.445645,5.1111,2026-10-16T01:58:18.333Z
